In [38]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append(str(Path().resolve().parent))

from pathlib import Path
from src import inputs, firms_api

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Visualise Wildfires (FIRMS API)
by Paul Ghisletti, May 18th 2026
***

## What is FIRMS?

**NASA FIRMS (Fire Information for Resource Management System)** provides near real-time satellite observations of active fires and thermal anomalies worldwide. The system aggregates wildfire detections from sensors such as MODIS and VIIRS and makes them accessible through APIs, maps, and downloadable datasets.

Typical FIRMS datasets contain:

- geographic coordinates (latitude and longitude)
- acquisition date and time
- fire radiative power (FRP)
- confidence scores
- satellite and instrument identifiers
- brightness temperature measurements
- detection type classifications
- optional metadata such as day/night flags and scan geometry

Each row usually represents a single satellite fire detection pixel rather than an individual wildfire event. Multiple nearby detections may therefore correspond to the same wildfire.
***
## Instructions
1. Set up your API-key with the instructions below.
2. Hit `Run All` to start running the cells in this notebook
3. Define the parameters of the API query by typing your desired inputs in the widgets below. The parameter are:
    - area: what area would you like the data to cover?
    - date: from when should the data be?
    - sensor: what sensor should provide the data?
    - number of days: what timespan should your data cover?
    - open in browser: you can decide whether the map should automaticall be opened in your browser.
4. Hit enter after each input to continue the process.
5. Once all parameters are set and you have read through the information about the map, hit enter to generate the map automatically.
***
## API-key Setup
Make sure you followed the steps 2.1. to 2.3. in the `README.md` file on how to set your individual API-key: Check, whether the following output matches your individual API-key from FIRMS. Here, you can also see how many free transactions you have left with your api-key

In [39]:
api_key = firms_api.get_api_key()
firms_api.api_key_status(api_key)

API-Key: 3d1ef73c3c932e5f3738d87f205a9cec


transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object

***
## Input Parameters
### 1. Area
There are 4 possible input types for this parameter:
- Country:
    - type a country's name e.g. 'Algeria'. Common alternatives should work too, e.g. 'Burma' for Myanmar.
    - ISO 3-letter country code, e.g. 'AUS' for Australia or 'GER' for Germany
- Continent: choose from one of the 7 continents.
- 'World': for global coverage, use this. This is also the default value if you do not input anything.
- Bounding Box: if you have a more specific area in mind, pass a list as input with the format: `[west, south, east, north]`

In [40]:
inputs.ask_area()

area input: africa


### 2. Date
Type your date in the format: **YYYY-MM-DD**. If you wish the **most recent data, leave this cell empty**.

In [41]:
inputs.ask_date()

date input: 


### 3. Sensor
There are 3 Satellite instruments offered by FIRMS, split into 4 sensors. For most sensors, there are 2 datasets, one for near real time data (usually current day) and standard processing (preprocessed by FIRMS, but containing more information. Latency usually months)
| Dataset          | Resolution | Coverage | revisit time        | Latency              | Best For                     |
| ---------------- | ---------- | -------- | ------------------- | -------------------- | ---------------------------- |
| LANDSAT_NRT      | 30 m       | US/CAN   | ~ 16 days           | Near real-time       | Detailed fire mapping        |
| MODIS_NRT        | 1 km       | Global   | ~ 0.25 days         | Near real-time       | Large active fires           |
| MODIS_SP         | 1 km       | Global   | ~ 0.25 days         | Standard processing  | Historical analysis          |
| VIIRS_NOAA20_NRT | 375 m      | Global   | ~ 0.5 days          | Near real-time       | Small/medium fires           |
| VIIRS_NOAA20_SP  | 375 m      | Global   | ~ 0.5 days          | Standard processing  | Higher-quality archive       |
| VIIRS_NOAA21_NRT | 375 m      | Global   | ~ 0.5 days          | Near real-time       | Newest VIIRS stream          |
| VIIRS_SNPP_NRT   | 375 m      | Global   | ~ 0.5 days          | Near real-time       | General wildfire monitoring  |
| VIIRS_SNPP_SP    | 375 m      | Global   | ~ 0.5 days          | Standard processing  | Historical wildfire analysis |  


Also note, that different sensors and datasets provide different time spans:

In [42]:
availability_all_df = firms_api.get_availability_all(api_key)

,data_id,min_date,max_date
0,MODIS_NRT,2026-03-01,2026-05-22
1,MODIS_SP,2000-11-01,2026-02-28
2,VIIRS_NOAA20_NRT,2026-04-01,2026-05-22
3,VIIRS_NOAA20_SP,2018-04-01,2026-03-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-22
5,VIIRS_SNPP_NRT,2026-04-01,2026-05-22
6,VIIRS_SNPP_SP,2012-01-20,2026-03-31
7,LANDSAT_NRT,2022-06-20,2026-05-21


For the sensor input, just copy paste one of the options from the table above:

In [43]:
inputs.ask_sensor()

sensor input: VIIRS_SNPP_SP	


### 4. Number of Days
The FIRMS API allows up to 5 days. The range begins with the date set above and counts n days to the future. However, nevermind if you put not date expecting the most recent data and now enter 5, since this contradiction is handled inside the functions.

In [44]:
inputs.ask_n_days()

n_days input: 5


### 5. Open in Browser
You can choose to open the .html file (which stores the visualisation data) in your browser for a larger panel:  
- `True`: Open in Browser (larger interface)
- `False`: Open in Notebook (smaller interface)
- If left empty, it will open in Notebook

In [45]:
inputs.ask_browser()

browser input: true


## Visualisation
### Map Layers:
- Area outlines
- Heatmap (weighted by FRP)
- Fires (clustered pixels)
- Severity Score
- Fire pixels by Fire Radiative Power in megawatts
- Fire pixels by detection time
- Volcanoes
- Offshore fires
- Other landbased fire sources  

Depending on the sensor/dataset you chose, only select layers are available and displayed:  
|                   |Area outlines  |Heatmap    |Fires (clustered)  |Severity score |Fire pixels by FRP |Fire pixels by datetime    |Volcanoes  |Offshore       |Other land source
|-------------------|---------------|-----------|-------------------|---------------|-------------------|---------------------------|-----------|---------------|----------
|LANDSAT_NRT        |       x       |     x     |         x         |               |                   |              x            |           |               |           
|MODIS_NRT          |       x       |     x     |         x         |               |         x         |              x            |           |               |           
|MODIS_SP           |       x       |     x     |         x         |               |         x         |              x            |     x     |       x       |     x     
|VIIRS_NOAA20_NRT   |       x       |     x     |         x         |               |         x         |              x            |           |               |           
|VIIRS_NOAA20_SP    |       x       |     x     |         x         |       x       |         x         |              x            |     x     |       x       |     x     
|VIIRS_NOAA21_NRT   |       x       |     x     |         x         |               |         x         |              x            |           |               |           
|VIIRS_SNPP_NRT     |       x       |     x     |         x         |               |         x         |              x            |           |               |           
|VIIRS_SNPP_SP      |       x       |     x     |         x         |       x       |         x         |              x            |     x     |       x       |     x    